In [1]:
from pathlib import Path

import json
import pandas as pd


# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw data
NUTRITION_RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nutrition"
    / "train-00000-of-00001.parquet"
)

FOODSEG_LABELS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "FoodSeg103"
    / "id2label.json"
)

# Processed data
NUTRITION_PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "nutrition"
)

NUTRITION_PROCESSED_PATH = (
    NUTRITION_PROCESSED_DIR
    / "nutrition_reference.csv"
)


print("Project root:", PROJECT_ROOT)
print("Nutrition raw dataset:", NUTRITION_RAW_PATH)
print("FoodSeg103 labels:", FOODSEG_LABELS_PATH)
print("Processed output:", NUTRITION_PROCESSED_PATH)

Project root: c:\Users\SANDEEP\Documents\NutriVision
Nutrition raw dataset: c:\Users\SANDEEP\Documents\NutriVision\data\raw\nutrition\train-00000-of-00001.parquet
FoodSeg103 labels: c:\Users\SANDEEP\Documents\NutriVision\data\raw\FoodSeg103\id2label.json
Processed output: c:\Users\SANDEEP\Documents\NutriVision\data\processed\nutrition\nutrition_reference.csv


In [2]:
nutrition_df = pd.read_parquet(NUTRITION_RAW_PATH)

print("Shape:", nutrition_df.shape)
print("\nColumns:")
print(nutrition_df.columns.tolist())

Shape: (103, 24)

Columns:
['class_id', 'class_name', 'food_description', 'g_per_metric_cup', 'density_g_per_cm3', 'density_kg_per_m3', 'calories_kcal', 'fat_total_g', 'fat_saturated_g', 'fat_polyunsat_g', 'fat_monounsat_g', 'fat_trans_g', 'cholesterol_mg', 'sodium_mg', 'carb_g', 'fiber_g', 'sugar_g', 'protein_g', 'vitamin_a_ug', 'vitamin_c_mg', 'vitamin_d_ug', 'calcium_mg', 'iron_mg', 'potassium_mg']


In [3]:
nutrition_df.head()

,class_id,class_name,food_description,g_per_metric_cup,density_g_per_cm3,density_kg_per_m3,calories_kcal,fat_total_g,fat_saturated_g,fat_polyunsat_g,...,carb_g,fiber_g,sugar_g,protein_g,vitamin_a_ug,vitamin_c_mg,vitamin_d_ug,calcium_mg,iron_mg,potassium_mg
0,0,French beans,"Beans, snap, green, raw",162.0,0.648,648.0,343.0,2.02,0.221,1.207,...,64.11,25.2,NaN,18.81,0.0,4.6,NaN,186.0,3.40,1316.0
1,1,almond,"Nuts, almonds, dry roasted, without salt added",146.0,0.584,584.0,578.0,50.64,3.881,12.214,...,19.74,11.8,4.80,21.26,0.0,0.0,NaN,248.0,4.30,728.0
2,2,apple,"Apple, raw",132.0,0.528,528.0,52.0,0.17,0.028,0.051,...,13.81,2.4,10.39,0.26,3.0,4.6,NaN,6.0,0.12,107.0
3,3,apricot,"Apricots, raw (halves)",164.0,0.656,656.0,48.0,0.39,0.027,0.077,...,11.12,2.0,9.24,1.40,96.0,10.0,NaN,13.0,0.39,259.0
4,4,asparagus,"Asparagus, raw (cut pieces)",142.0,0.568,568.0,20.0,0.12,0.046,0.088,...,3.88,2.1,1.88,2.20,38.0,5.6,NaN,24.0,2.14,202.0


In [4]:
with open(FOODSEG_LABELS_PATH, "r", encoding="utf-8") as f:
    foodseg_id2label = json.load(f)

# Convert JSON keys from strings to integers
foodseg_id2label = {
    int(class_id): class_name.strip()
    for class_id, class_name in foodseg_id2label.items()
}

print("Total FoodSeg103 labels:", len(foodseg_id2label))
print("Background:", foodseg_id2label[0])
print("Food classes:", len(foodseg_id2label) - 1)

print("\nFirst 10 food classes:")
for class_id in sorted(foodseg_id2label)[1:11]:
    print(f"{class_id}: {foodseg_id2label[class_id]}")

Total FoodSeg103 labels: 104
Background: background
Food classes: 103

First 10 food classes:
1: candy
2: egg tart
3: french fries
4: chocolate
5: biscuit
6: popcorn
7: pudding
8: ice cream
9: cheese butter
10: cake


In [5]:
# Normalize names only for comparison.
# This does not modify the original datasets.
foodseg_names = {
    class_name.strip().lower()
    for class_id, class_name in foodseg_id2label.items()
    if class_id != 0
}

nutrition_names = (
    nutrition_df["class_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

nutrition_name_set = set(nutrition_names)

missing_from_nutrition = sorted(foodseg_names - nutrition_name_set)
extra_in_nutrition = sorted(nutrition_name_set - foodseg_names)

duplicate_count = nutrition_names.duplicated().sum()

print("FoodSeg103 food classes:", len(foodseg_names))
print("Nutrition classes:", len(nutrition_name_set))
print("Nutrition duplicate names:", duplicate_count)
print("Missing from nutrition:", missing_from_nutrition)
print("Extra in nutrition:", extra_in_nutrition)

FoodSeg103 food classes: 103
Nutrition classes: 103
Nutrition duplicate names: 0
Missing from nutrition: []
Extra in nutrition: []


In [6]:
required_nutrition_columns = [
    "calories_kcal",
    "protein_g",
    "carb_g",
    "fat_total_g",
    "fiber_g",
]

print("Required nutrition columns:")
for column in required_nutrition_columns:
    print(f"- {column}")

print("\nMissing values:")
missing_required = nutrition_df[required_nutrition_columns].isna().sum()
print(missing_required)

Required nutrition columns:
- calories_kcal
- protein_g
- carb_g
- fat_total_g
- fiber_g

Missing values:
calories_kcal    0
protein_g        0
carb_g           0
fat_total_g      0
fiber_g          1
dtype: int64


In [7]:
missing_fiber = nutrition_df.loc[
    nutrition_df["fiber_g"].isna(),
    ["class_id", "class_name", "food_description", "fiber_g"],
]

print(missing_fiber.to_string(index=False))

 class_id class_name                                food_description  fiber_g
       43    hamburg Fast foods, hamburger, large, single meat patty      NaN


In [8]:
# Create a reverse mapping:
# normalized FoodSeg103 class name → canonical FoodSeg103 class ID

foodseg_name_to_id = {
    class_name.strip().lower(): class_id
    for class_id, class_name in foodseg_id2label.items()
    if class_id != 0
}

# Copy only the fields required from the raw nutrition dataset.
nutrition_reference = nutrition_df[
    [
        "class_name",
        "calories_kcal",
        "protein_g",
        "carb_g",
        "fat_total_g",
        "fiber_g",
    ]
].copy()

# Map each nutrition class name to its canonical FoodSeg103 ID.
nutrition_reference["foodseg103_class_id"] = (
    nutrition_reference["class_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(foodseg_name_to_id)
)

# Put the canonical ID first.
nutrition_reference = nutrition_reference[
    [
        "foodseg103_class_id",
        "class_name",
        "calories_kcal",
        "protein_g",
        "carb_g",
        "fat_total_g",
        "fiber_g",
    ]
]

print("Shape:", nutrition_reference.shape)
nutrition_reference.head()

Shape: (103, 7)


,foodseg103_class_id,class_name,calories_kcal,protein_g,carb_g,fat_total_g,fiber_g
0,96,French beans,343.0,18.81,64.11,2.02,25.2
1,17,almond,578.0,21.26,19.74,50.64,11.8
2,25,apple,52.0,0.26,13.81,0.17,2.4
3,27,apricot,48.0,1.40,11.12,0.39,2.0
4,85,asparagus,20.0,2.20,3.88,0.12,2.1


In [9]:
expected_columns = [
    "foodseg103_class_id",
    "class_name",
    "calories_kcal",
    "protein_g",
    "carb_g",
    "fat_total_g",
    "fiber_g",
]

assert nutrition_reference.shape == (103, 7)

assert nutrition_reference.columns.tolist() == expected_columns

assert nutrition_reference["foodseg103_class_id"].notna().all()

assert nutrition_reference["foodseg103_class_id"].nunique() == 103

assert nutrition_reference["class_name"].nunique() == 103

for column in required_nutrition_columns:
    assert pd.api.types.is_numeric_dtype(nutrition_reference[column])

print("Processed nutrition reference validation passed.")
print("Shape:", nutrition_reference.shape)
print("Unique FoodSeg103 IDs:", nutrition_reference["foodseg103_class_id"].nunique())
print("Unique class names:", nutrition_reference["class_name"].nunique())

Processed nutrition reference validation passed.
Shape: (103, 7)
Unique FoodSeg103 IDs: 103
Unique class names: 103


In [11]:
nutrition_reference[
    nutrition_reference["class_name"].str.lower() == "hamburg"
]

,foodseg103_class_id,class_name,calories_kcal,protein_g,carb_g,fat_total_g,fiber_g
43,60,hamburg,305.0,13.69,33.9,13.13,NaN


In [13]:
NUTRITION_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

nutrition_reference.to_csv(
    NUTRITION_PROCESSED_PATH,
    index=False,
)

print("Saved processed nutrition reference:")
print(NUTRITION_PROCESSED_PATH)

Saved processed nutrition reference:
c:\Users\SANDEEP\Documents\NutriVision\data\processed\nutrition\nutrition_reference.csv


In [14]:
saved_reference = pd.read_csv(NUTRITION_PROCESSED_PATH)

print("Saved shape:", saved_reference.shape)
print("Saved columns:")
print(saved_reference.columns.tolist())

Saved shape: (103, 7)
Saved columns:
['foodseg103_class_id', 'class_name', 'calories_kcal', 'protein_g', 'carb_g', 'fat_total_g', 'fiber_g']


In [15]:
saved_reference[
    saved_reference["class_name"].str.lower() == "hamburg"
]

,foodseg103_class_id,class_name,calories_kcal,protein_g,carb_g,fat_total_g,fiber_g
43,60,hamburg,305.0,13.69,33.9,13.13,NaN


In [16]:
final_reference = pd.read_csv(NUTRITION_PROCESSED_PATH)

assert final_reference.shape == (103, 7)

assert final_reference.columns.tolist() == expected_columns

assert final_reference["foodseg103_class_id"].nunique() == 103

assert final_reference["class_name"].nunique() == 103

hamburg_row = final_reference[
    final_reference["class_name"].str.lower() == "hamburg"
]

assert len(hamburg_row) == 1
assert pd.isna(hamburg_row["fiber_g"].iloc[0])

print("Final nutrition reference validation passed.")
print("Shape:", final_reference.shape)
print("Unique FoodSeg103 IDs:", final_reference["foodseg103_class_id"].nunique())
print("Unique class names:", final_reference["class_name"].nunique())
print("\nHamburg record:")
print(hamburg_row.to_string(index=False))

Final nutrition reference validation passed.
Shape: (103, 7)
Unique FoodSeg103 IDs: 103
Unique class names: 103

Hamburg record:
 foodseg103_class_id class_name  calories_kcal  protein_g  carb_g  fat_total_g  fiber_g
                  60    hamburg          305.0      13.69    33.9        13.13      NaN
